# Demo kỹ thuật — BPMN Công ty X

Notebook đọc **kết quả đã chạy**, không tạo số liệu giả và không tự huấn luyện lại. Đây là demo code để kiểm tra dữ liệu/kết quả; chưa phải slide hoặc report. Chạy `uv run --locked congtyx all` trước. Các giới hạn và công thức đầy đủ ở README.md.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
from bpmn_project.common import ROOT
OUT = ROOT / "outputs"
status = json.loads((OUT / "metadata/run_all.json").read_text())
assert status["status"] == "success" and not status["quick"]
pd.set_option("display.max_columns", 20)
print("Pipeline:", status["status"], "—", round(status["elapsed_seconds"], 1), "giây")
def table(name):
    return pd.read_csv(OUT / name)
def figure(name):
    display(Image(filename=str(OUT / "figures" / name), width=1000))

## 1. Dữ liệu và audit
Mỗi issue có mã case, event_id và dòng A1 để kiểm tra lại. Có log được chọn có chủ đích, thiếu ready_time và một số mốc kết thúc.

In [ ]:
display(pd.Series(json.loads((OUT / "audit/summary.json").read_text())))
display(table("audit/issues.csv").head(15))
display(table("audit/excel_control_differences.csv"))

## 2. Bằng chứng điểm nghẽn
WIP tổng thể, làm lại tổng thể và thời gian mẫu có mẫu số khác nhau. Quy tắc sàng lọc minh bạch; chưa chứng minh giới hạn công suất.

In [ ]:
display(table("process/bottleneck_evidence.csv").round(3))
display(table("process/case_metrics.csv").round(3))
figure("02_task_times.png")

## 3. Lỗi, lớp sơn và lượt lặp
Phân biệt chuyển sang lớp kế tiếp với làm lại trong cùng lớp. Tỷ lệ PASS là theo lượt kiểm tra quan sát trong mẫu 8 case.

In [ ]:
display(table("process/qc_outcomes.csv").round(3))
display(table("process/qc_by_layer.csv").round(3))
display(table("process/defect_pareto.csv").head(8).round(3))
figure("03_defects.png")

## 4. Process Mining và conformance
Discovery/replay cùng mẫu chỉ mô tả dữ liệu. Kiểm tra cấu trúc cục bộ không đánh giá guard. BPMN tổng thể chưa chứa Đóng gói/Nhập kho như Hình 1; không coi các event ngoài phạm vi này là vi phạm nghiệp vụ.

In [ ]:
display(table("mining/discovery_quality.csv").round(3))
display(table("mining/supplied_model_token_replay.csv").round(3))
display(table("mining/structural_deviations.csv").head(10))
figure("10_observed_dfg.png")

## 5. Feature, nhãn và holdout thời gian
Dự báo cuối t cho t+1; test feature tháng 07/2020–03/2021, target 08/2020–04/2021. Nhãn calendar_v1 được định nghĩa lại rõ ràng nên không so điểm trực tiếp với E1 cũ.

In [ ]:
contract = json.loads((OUT / "forecast/feature_contract.json").read_text())
print(json.dumps(contract["labels"], ensure_ascii=False, indent=2))
print("Khác nhãn Excel:", contract["label_comparison"])
display(table("forecast/model_comparison.csv").round(3))
display(table("forecast/ensemble_metrics_month_bootstrap.csv").round(3))

## 6. Graph ML và cảnh báo
Routing snapshot chưa có lịch sử phiên bản; kết quả graph có điều kiện topology đã được biết tại cutoff. PCA là hình mô tả, không phải bằng chứng dự báo tốt. Chính sách cảnh báo minh họa dùng RF WIP, không chọn mô hình thắng test để triển khai.

In [ ]:
figure("04_models_y_surge_next.png")
figure("08_node2vec_projection.png")
alerts = table("forecast/alert_review_policy.csv")
display(alerts[alerts.review_alert].head(10))

## 7. Kịch bản cải tiến
Ba đầu ra riêng: replay khoảng quan sát, khối lượng làm lại và timer QC7. Không cộng chúng thành mức giảm Cycle Time toàn nhà máy. Mức giảm gap/rework và chi phí checklist là giả định cần kiểm chứng.

In [ ]:
display(table("scenarios/gap_replay_summary.csv").round(3))
display(table("scenarios/repeat_workload_sensitivity.csv").round(3))
display(table("scenarios/qc7_timer_dispatch_replay.csv").round(3))
figure("07_scenario_sensitivity.png")

## 8. Tái lập
`uv.lock`, `config.toml`, hash input/source code, test và danh mục artifact đi kèm kết quả. Giữ tên file/phiên bản nhãn khi trích số liệu sang report sau này.

In [ ]:
manifest = json.loads((OUT / "metadata/manifest.json").read_text())
display(pd.DataFrame(manifest["sources"]))
display(table("metadata/artifact_inventory.csv").head(15))